# KHUDA 10기 ML세션
Date : 2026.07.29 (수)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import f1_score, recall_score

In [ ]:
# dataset repository path

url = "https://github.com/chaegyeong/KHUDA_ML_10th/raw/cf22e3b419f827b39713e3f3c2db73d4caa1626c/KHUDA_ML10th_WEEK3/ML10th_WEEK3.csv"

In [ ]:
df = pd.read_csv(url)
df.head(5)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,39972.0,1.264068,1.274004,-1.953181,1.394859,1.484954,-0.867329,0.721149,-0.228330,-0.483248,...,-0.231105,-0.496493,-0.289724,-1.075149,0.916810,-0.214503,0.059242,0.086327,1.00,0
1,132349.0,2.004888,0.139182,-1.852629,0.325115,0.476550,-0.868723,0.208379,-0.235157,0.266502,...,0.242830,0.844270,-0.095814,-0.512939,0.275697,-0.094631,-0.002533,-0.042010,18.54,0
2,168475.0,1.819787,-0.133835,-1.823455,0.619407,0.117000,-1.386072,0.453325,-0.431470,0.495223,...,0.279400,0.746038,-0.124563,-0.056314,0.212661,-0.112755,-0.018110,-0.009893,122.00,0
3,52094.0,-1.852604,0.241480,1.722107,1.042119,-0.903545,0.880198,0.168185,0.598554,0.112700,...,-0.046677,0.278853,0.368968,-0.448355,0.345620,-0.185181,0.007258,-0.138822,165.28,0
4,75756.0,1.125023,0.253599,0.268421,1.007613,0.154839,0.144089,0.055003,0.034491,-0.435384,...,0.156214,0.502788,-0.167968,-0.258314,0.645635,-0.254973,0.033560,0.011955,28.56,0


In [ ]:
### 건들지마세요
n=4
answer = [0] * 5

## Task 0

초기 설정을 진행합니다.

다음 설정에 맞게 데이터셋을 분리해주세요.
별도의 정답은 없지만, 완료되지 않은 경우 Task 수행 불가

* feature(X) : Class를 제외한 전체 컬럼 [Time, V1~V28, Amount] - drop 사용을 추천합니다.
이때 Amount 컬럼만 StandardScaler로 스케일링을 진행합니다.

* target(y) : Class

* train_test_split : test_size=0.2, random_state=42, stratify=y


In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

scaler = StandardScaler()
X['Amount'] = scaler.fit_transform(X[['Amount']])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Task 1

DecisionTreeClassifier(random_state=42)를 기준으로,GridSearchCV(cv=3, scoring='f1')로 최적 하이퍼파라미터를 탐색하세요.

max_depth 후보: [4, 6, 8, 10]

min_samples_split 후보: [2, 3, 4]

탐색된 최적 max_depth와 min_samples_split 두 값의 합을 answer[1]에 저장하세요.


* Hint : best_params_ 를 사용하면 최적의 파라미터를 반환합니다

In [ ]:
## Input Box

ans = 0


dt_clf = DecisionTreeClassifier(random_state=42)
param_grid = {
    'max_depth': [4, 6, 8, 10],
    'min_samples_split': [2, 3, 4]
}

grid_dt = GridSearchCV(dt_clf, param_grid=param_grid, cv=3, scoring='f1')
grid_dt.fit(X_train, y_train)

best_depth = grid_dt.best_params_['max_depth']
best_split = grid_dt.best_params_['min_samples_split']

ans = best_depth + best_split

In [ ]:
answer[1] = ans
print(answer[1])

6


## Task 2

RandomForestClassifier(n_estimators=100, random_state=42)로 학습을 진행합니다. test 데이터에 대한 사기 거래(Class=1)의 재현율(recall)을 (소수점 셋째 자리에서 반올림) 둘째자리까지 구하세요. 해당 값을 answer[2]에 저장하세요.


In [ ]:
## Input Box

ans = 0

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)
rf_pred = rf_clf.predict(X_test)

ans= round(recall_score(y_test, rf_pred), 2)

In [ ]:
answer[2] = ans
print(answer[2])

0.88


## Task 3


Task 2에서 학습한 모델의(RandomForestClassifier를 적용한 모델) 피처 중요도가 0.07 이상인 변수의 총 개수를 구해 answer[3]에 저장하세요.

In [ ]:
## Input Box

ans = 0

importances = rf_clf.feature_importances_

ans = np.sum(importances >= 0.07)

In [ ]:
answer[3] = ans
print(answer[3])

5


## Task 4
Task 3에서 구한 상위 중요 변수(피처 중요도가 0.07 이상인 변수)만 활용해서 RandomForestClassifier(n_estimators=100, random_state=42)를 새로 학습하세요.

test 데이터에 대한 사기 거래(Class=1)의 f1-score을 (소수점 넷째 자리에서 반올림) 셋째자리까지 구하세요. 해당 값을 answer[4]에 저장하세요.

* Hint : DataFrame loc[] 연산자 를 활용합니다. (자세한 내용은 교재 파머완 p67~ 을 참고해주세요.)
* Optional : 만약, 선택된 컬럼들의 이름을 보고싶다면 .columns 을 사용하면 됩니다. (사용하지 않아도 문제 해결에 지장 없음)

In [ ]:
## 아래는 .columns 예시입니다
# 피처 중요도가 0.07 이상인 컬럼 확인하기
# 아래 주석 코드가 없어도 해결 가능한 문제입니다
# 혹시 선택된 컬럼이 뭔지 궁금하다면 이런식으로 확인 가능하다고 알려주시면 됩니다
# top_features = X_train.columns[importances >= 0.07]
# print(list(top_columns))


## Input Box

ans = 0

importances = rf_clf.feature_importances_
X_train_top = X_train.loc[:, importances >= 0.07]
X_test_top = X_test.loc[:, importances >= 0.07]


rf_top = RandomForestClassifier(n_estimators=100, random_state=42)
rf_top.fit(X_train_top, y_train)


pred_selected = rf_top.predict(X_test_top)
ans = round(f1_score(y_test, pred_selected), 3)



In [ ]:
answer[4] = ans
print(answer[4])

0.925


# 정답 확인!

In [ ]:
print("=== 작성하신 정답 ===")
for i in range (1, (n+1)) : print("Task " + str(i) + " :: " + str(answer[i]))

=== 작성하신 정답 ===
Task 1 :: 6
Task 2 :: 0.88
Task 3 :: 5
Task 4 :: 0.925


In [ ]:
### 정답 채점 코드
import hashlib

ANSWER_URL = "https://github.com/chaegyeong/KHUDA_ML_10th/raw/d965d66675ef573cac63fb7ec88e6824080ed644/KHUDA_ML10th_WEEK3/answer.csv"

def md5 (s) : return hashlib.md5(s.encode("utf-8")).hexdigest()

ans_df = pd.read_csv(ANSWER_URL)
gt = {int(r["task"]): str(r["answer"]).strip().lower()
      for _, r in ans_df.iterrows()}

wrong = []

for i in range(1, (n+1)) :
    user_answer = "" if answer[i] is None else md5(str(answer[i]).strip().lower())
    if (user_answer != gt.get(i, "")) : wrong.append(i)

print("탈출!" if not wrong else f"틀린번호 : {wrong}")

탈출!
